In [8]:
# Significant similarity to audio descriptions. 

import numpy as np
import pandas as pd

scores = np.load('scores/S1/perceived_movie/presto.npz', allow_pickle=True)
print(scores.files)
score_names = ['window_scores', 'window_zscores', 'story_scores', 'story_zscores']
score_files = {}
for name in score_names:
    score_files[name] = scores[name].item()

print(score_files['window_zscores'])

['window_scores', 'window_zscores', 'story_scores', 'story_zscores']
{('presto', 'WER'): array([ 0.32187847,  0.56803756, -0.01503934,  0.19242198, -0.32189659,
        0.23068393,  0.42608143,  0.19448613,  0.47821188,  0.23526341,
        0.88178405, -0.3890192 , -1.21424986, -1.39585843, -1.58178445,
       -0.84706079, -1.05826618,  0.21744906,  0.05098364, -1.20463479,
       -0.9426986 , -0.80050187, -0.88099419, -0.90453403, -1.04755938,
       -0.9910207 , -0.99649191, -0.78539647, -0.8781085 , -0.86643552,
       -0.34237614, -0.53970867, -0.01265575,  0.20282414,  0.51121335,
        0.07398817, -0.44231287,  0.6423086 ,  0.58419   ,  1.13499697,
        0.4872186 ,  0.10624879, -0.08391814, -0.00916737, -0.95941353,
       -0.80060249, -0.1533576 , -0.7097151 , -0.86840753, -0.99138448,
       -0.75645585, -0.61034485,  0.55991439,  0.43154766,  0.89281533,
        1.23069233,  1.4667176 ,  1.40615645,  2.24427351,  3.10873377,
        3.0567028 ,  2.63779803,  3.57548854,  

In [24]:
window_zscores = {'subject': [], 'WER':[],'BLEU':[], 'METEOR':[], 'BERT':[]}
for subject in [1,2,3]:
    for task in ['presto','partlycloudy','laluna']:
        scores = np.load(f'scores/S{subject}/perceived_movie/{task}.npz', allow_pickle=True)['window_zscores'].item()
        # print(scores['window_zscores'].item())
        window_zscores['subject'].append(subject)
        window_zscores['WER'].append(scores[(task, 'WER')])
        window_zscores['BLEU'].append(scores[(task, 'BLEU')])
        window_zscores['METEOR'].append(scores[(task, 'METEOR')])
        window_zscores['BERT'].append(scores[(task, 'BERT')])

window_zscores

{'subject': [1, 1, 1, 2, 2, 2, 3, 3, 3],
 'WER': [array([ 0.32187847,  0.56803756, -0.01503934,  0.19242198, -0.32189659,
          0.23068393,  0.42608143,  0.19448613,  0.47821188,  0.23526341,
          0.88178405, -0.3890192 , -1.21424986, -1.39585843, -1.58178445,
         -0.84706079, -1.05826618,  0.21744906,  0.05098364, -1.20463479,
         -0.9426986 , -0.80050187, -0.88099419, -0.90453403, -1.04755938,
         -0.9910207 , -0.99649191, -0.78539647, -0.8781085 , -0.86643552,
         -0.34237614, -0.53970867, -0.01265575,  0.20282414,  0.51121335,
          0.07398817, -0.44231287,  0.6423086 ,  0.58419   ,  1.13499697,
          0.4872186 ,  0.10624879, -0.08391814, -0.00916737, -0.95941353,
         -0.80060249, -0.1533576 , -0.7097151 , -0.86840753, -0.99138448,
         -0.75645585, -0.61034485,  0.55991439,  0.43154766,  0.89281533,
          1.23069233,  1.4667176 ,  1.40615645,  2.24427351,  3.10873377,
          3.0567028 ,  2.63779803,  3.57548854,  4.2765146 ,  4.

In [52]:
results_df = pd.DataFrame(window_zscores)

print(len(results_df.loc[0, 'WER']))
print(len(results_df.loc[1, 'WER']))
print(len(results_df.loc[2, 'WER']))
print('=')
print(235+263+331)
m=829

from scipy.stats import norm
#convert to p val
results_df['WER'] = results_df['WER'].apply(lambda l : [1-norm.cdf(z) for z in l])
results_df['BLEU'] = results_df['BLEU'].apply(lambda l : [1-norm.cdf(z) for z in l])
results_df['METEOR'] = results_df['METEOR'].apply(lambda l : [1-norm.cdf(z) for z in l])
results_df['BERT'] = results_df['BERT'].apply(lambda l : [1-norm.cdf(z) for z in l])
#sort
for i in range(9):
    for met in ['WER','BLEU','METEOR','BERT']:
        results_df.loc[i,met].sort()
# to q and check threshold
def p_to_q(p, i):
    return p*m/i
results_df['WER'] = results_df['WER'].apply(lambda l : [1 if p_to_q(l[i-1], i) < 0.05 else 0 for i in range(1, len(l)+1)])
results_df['BLEU'] = results_df['BLEU'].apply(lambda l : [1 if p_to_q(l[i-1], i) < 0.05 else 0 for i in range(1, len(l)+1)])
results_df['METEOR'] = results_df['METEOR'].apply(lambda l : [1 if p_to_q(l[i-1], i) < 0.05 else 0 for i in range(1, len(l)+1)])
results_df['BERT'] = results_df['BERT'].apply(lambda l : [1 if p_to_q(l[i-1], i) < 0.05 else 0 for i in range(1, len(l)+1)])

S1=np.array(results_df.loc[0, 'BERT'] + results_df.loc[1, 'BERT'] + results_df.loc[2, 'BERT']).mean()
S2=np.array(results_df.loc[3, 'BERT'] + results_df.loc[4, 'BERT'] + results_df.loc[5, 'BERT']).mean()
S3=np.array(results_df.loc[6, 'BERT'] + results_df.loc[7, 'BERT'] + results_df.loc[8, 'BERT']).mean()
print(S1,S2,S3,'BERT')

results_df

235
263
331
=
829
0.1158021712907117 0.39324487334137515 0.29191797346200243 BERT


,subject,WER,BLEU,METEOR,BERT
0,1,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1,1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,1,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3,2,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
4,2,"[1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
5,2,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
6,3,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
7,3,"[1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
8,3,"[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."


In [54]:
to_file = pd.DataFrame({'subject':[1,2,3], 'significantly_decoded': [S1,S2,S3]})
to_file.to_csv('perceived_movie_percentages.csv', index=False)

to_file

,subject,significantly_decoded
0,1,0.115802
1,2,0.393245
2,3,0.291918


In [27]:
results = np.load('results/S1/perceived_speech/wheretheressmoke.npz', allow_pickle=True)
result_names = results.files
result_files={}
for name in result_names:
    result_files[name] = results[name]
result_files

{'words': array(['she', 'said', 'she', ..., 'that', 'she', 'needs'],
       shape=(1589,), dtype='<U13'),
 'times': array([ 10.2,  10.6,  11. , ..., 591. , 591.4, 591.8], shape=(1589,))}